# Path 4: Error / edge-case paths + full teardown

Exercises the negative paths every resource (`customers`, `products`, `orders`,
`order-items`) shares: invalid ids, missing required fields, invalid enum
values, and not-found lookups. Also asserts that deleting a product still
referenced by an order-item returns 409. Finishes with a full teardown of a
purpose-built customer/order/item chain, verifying 404s after delete
(exercising the DB `ON DELETE CASCADE` chain too).

Run top-to-bottom (e.g. `jupyter nbconvert --to notebook --execute 04_cleanup_and_error_paths.ipynb`).

In [1]:
import os

import requests
from faker import Faker

BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:3002/api")
fake = Faker()
NONEXISTENT_ID = 999_999_999

print(f"BASE_URL={BASE_URL}")

BASE_URL=http://api-service:5000/api


## Step 1 - Invalid id (non-numeric) returns 400 on every resource

In [2]:
for resource in ["customers", "products", "orders", "order-items"]:
    resp = requests.get(f"{BASE_URL}/{resource}/not-a-number")
    assert resp.status_code == 400, f"{resource}: expected 400, got {resp.status_code}"

    resp = requests.delete(f"{BASE_URL}/{resource}/not-a-number")
    assert resp.status_code == 400, f"{resource}: expected 400, got {resp.status_code}"

print("Confirmed 400 for invalid ids across customers/products/orders/order-items")

Confirmed 400 for invalid ids across customers/products/orders/order-items


## Step 2 - Not-found id (well-formed but nonexistent) returns 404 on every resource

In [3]:
for resource in ["customers", "products", "orders", "order-items"]:
    resp = requests.get(f"{BASE_URL}/{resource}/{NONEXISTENT_ID}")
    assert resp.status_code == 404, f"{resource}: expected 404, got {resp.status_code}"

resp = requests.get(f"{BASE_URL}/customers/{NONEXISTENT_ID}/orders")
assert resp.status_code == 404, resp.text

resp = requests.get(f"{BASE_URL}/orders/{NONEXISTENT_ID}/items")
assert resp.status_code == 404, resp.text

print("Confirmed 404 for nonexistent ids, including relation sub-routes")

Confirmed 404 for nonexistent ids, including relation sub-routes


## Step 3 - Missing required fields returns 400 (customers, products, orders, order-items)

In [4]:
# customers: email/password required
resp = requests.post(f"{BASE_URL}/customers", json={"email": fake.email()})
assert resp.status_code == 400, resp.text
resp = requests.post(f"{BASE_URL}/customers", json={"password": fake.password()})
assert resp.status_code == 400, resp.text

# products: sku, name, unit_price_cents required
resp = requests.post(f"{BASE_URL}/products", json={"sku": "SKU-MISSING"})
assert resp.status_code == 400, resp.text

# orders: customer_id required
resp = requests.post(f"{BASE_URL}/orders", json={})
assert resp.status_code == 400, resp.text

# order-items: order_id, product_id, quantity, unit_price_cents required
resp = requests.post(f"{BASE_URL}/order-items", json={"order_id": 1})
assert resp.status_code == 400, resp.text

print("Confirmed 400 for missing required fields")

Confirmed 400 for missing required fields


## Step 4 - Invalid `status` enum on orders returns 400

In [5]:
resp = requests.post(
    f"{BASE_URL}/orders",
    json={"customer_id": 1, "status": "not-a-real-status"},
)
assert resp.status_code == 400, resp.text
print("Confirmed 400 for invalid order status on create")

Confirmed 400 for invalid order status on create


## Step 5 - Build a small chain to tear down: customer -> order -> order-item

In [6]:
resp = requests.post(
    f"{BASE_URL}/customers",
    json={"email": fake.unique.email(), "password": fake.password(length=14)},
)
assert resp.status_code == 201, resp.text
customer = resp.json()

resp = requests.post(
    f"{BASE_URL}/products",
    json={
        "sku": f"SKU-{fake.unique.bothify(text='???-####').upper()}",
        "name": fake.unique.catch_phrase(),
        "unit_price_cents": 999,
    },
)
assert resp.status_code == 201, resp.text
product = resp.json()

resp = requests.post(f"{BASE_URL}/orders", json={"customer_id": customer["id"]})
assert resp.status_code == 201, resp.text
order = resp.json()

resp = requests.post(
    f"{BASE_URL}/order-items",
    json={
        "order_id": order["id"],
        "product_id": product["id"],
        "quantity": 1,
        "unit_price_cents": product["unit_price_cents"],
    },
)
assert resp.status_code == 201, resp.text
item = resp.json()

print(
    f"Built chain: customer={customer['id']} product={product['id']} "
    f"order={order['id']} order_item={item['id']}"
)

Built chain: customer=218 product=3729 order=4009 order_item=3016


## Step 6 - DELETE /products/:id while referenced returns 409

In [7]:
resp = requests.delete(f"{BASE_URL}/products/{product['id']}")
assert resp.status_code == 409, f"expected 409 while referenced, got {resp.status_code} {resp.text}"
print("Confirmed 409: product cannot be deleted while order-items reference it")

Confirmed 409: product cannot be deleted while order-items reference it


## Step 7 - Explicit teardown: delete item -> product -> order -> customer, verifying 404s

In [8]:
resp = requests.delete(f"{BASE_URL}/order-items/{item['id']}")
assert resp.status_code == 200, resp.text
resp = requests.get(f"{BASE_URL}/order-items/{item['id']}")
assert resp.status_code == 404, resp.text

resp = requests.delete(f"{BASE_URL}/products/{product['id']}")
assert resp.status_code == 200, resp.text
resp = requests.get(f"{BASE_URL}/products/{product['id']}")
assert resp.status_code == 404, resp.text

resp = requests.delete(f"{BASE_URL}/orders/{order['id']}")
assert resp.status_code == 200, resp.text
resp = requests.get(f"{BASE_URL}/orders/{order['id']}")
assert resp.status_code == 404, resp.text

resp = requests.delete(f"{BASE_URL}/customers/{customer['id']}")
assert resp.status_code == 200, resp.text
resp = requests.get(f"{BASE_URL}/customers/{customer['id']}")
assert resp.status_code == 404, resp.text

print("Teardown complete: item, product, order, and customer all confirmed gone (404)")

Teardown complete: item, product, order, and customer all confirmed gone (404)


## Step 8 - Cascade delete: deleting a customer with an order removes its order-items (product stays)

In [9]:
resp = requests.post(
    f"{BASE_URL}/customers",
    json={"email": fake.unique.email(), "password": fake.password(length=14)},
)
assert resp.status_code == 201, resp.text
cascade_customer = resp.json()

resp = requests.post(
    f"{BASE_URL}/products",
    json={
        "sku": f"SKU-{fake.unique.bothify(text='???-####').upper()}",
        "name": fake.unique.catch_phrase(),
        "unit_price_cents": 500,
    },
)
assert resp.status_code == 201, resp.text
cascade_product = resp.json()

resp = requests.post(f"{BASE_URL}/orders", json={"customer_id": cascade_customer["id"]})
assert resp.status_code == 201, resp.text
cascade_order = resp.json()

resp = requests.post(
    f"{BASE_URL}/order-items",
    json={
        "order_id": cascade_order["id"],
        "product_id": cascade_product["id"],
        "quantity": 2,
        "unit_price_cents": cascade_product["unit_price_cents"],
    },
)
assert resp.status_code == 201, resp.text
cascade_item = resp.json()

# Deleting the customer directly should cascade through orders -> order_items in the DB.
resp = requests.delete(f"{BASE_URL}/customers/{cascade_customer['id']}")
assert resp.status_code == 200, resp.text

resp = requests.get(f"{BASE_URL}/orders/{cascade_order['id']}")
assert resp.status_code == 404, "order should be gone after cascading customer delete"

resp = requests.get(f"{BASE_URL}/order-items/{cascade_item['id']}")
assert resp.status_code == 404, "order-item should be gone after cascading customer delete"

resp = requests.get(f"{BASE_URL}/products/{cascade_product['id']}")
assert resp.status_code == 200, "product should remain after customer cascade (ON DELETE RESTRICT)"

resp = requests.delete(f"{BASE_URL}/products/{cascade_product['id']}")
assert resp.status_code == 200, resp.text

print("Confirmed ON DELETE CASCADE removes orders and order-items; product catalog row is independent")

Confirmed ON DELETE CASCADE removes orders and order-items; product catalog row is independent
